# Lecture 08: Diagnosing & Mitigating Data Skew in Distributed Systems
This notebook guides you through simulating a data skew bottleneck where a single executor thread gets overwhelmed by a massive key partition, and resolving it using Adaptive Query Execution (AQE) and the **Salting Pattern**.

### 1. Initialize SparkSession
We configure the session with local cores and set shuffle partition parameters.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lecture08_DataSkewSalting") \
    .master("local[2]") \
    .getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark Session initialized successfully!")

### 2. Configure Shuffle Partition Count dynamically
Setting shuffle partition parameters to control partition count during aggregations and joins.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("Shuffle partitions set to 8.")

### 3. Activating Adaptive Query Execution (AQE)
Enabling AQE which allows Spark to optimize execution plans at runtime based on actual partition metrics.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("Adaptive Query Execution (AQE) enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

### 4. Activating AQE Skew Join Optimization
Enabling automated skew join optimization. When Spark detects a skewed partition, it splits it into smaller sub-partitions and joins them with duplicated lookup keys.

In [ ]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
print("AQE Skew Join optimization enabled:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

### 5. Simulate Skewed Transactions Dataset
We generate a dataset where 90% of transactions belong to a single merchant ID ('M_1001'), creating a massive partition key bottleneck.

In [ ]:
from pyspark.sql.functions import col, when

skewed_df = spark.range(1, 200000).withColumn(
    "merchantId",
    when(col("id") % 10 != 0, "M_1001") \
    .when(col("id") % 20 == 0, "M_1002") \
    .otherwise("M_1003")
).withColumn("amount", col("id") * 0.5)

skewed_df.groupBy("merchantId").count().show()

### 6. Create Merchant Lookup Dimension Dataset
Creating a lookup table that maps merchant IDs to names.

In [ ]:
merchants_df = spark.createDataFrame([
    ("M_1001", "MegaStore_Inc"),
    ("M_1002", "LocalShop_LLC"),
    ("M_1003", "Cafe_Corner")
], ["merchantId", "merchantName"])
merchants_df.show()

### 7. Unoptimized Skewed Join (Triggers slow executor task execution)
Executing a join on the skewed key. In a multi-node cluster, the executor handling 'M_1001' will take significantly longer to finish (task straggler).

In [ ]:
join_unoptimized = skewed_df.join(merchants_df, "merchantId")
print("Unoptimized join query constructed.")

### 8. Explain Plan of Unoptimized Join
Inspecting the physical plan to identify exchange operations.

In [ ]:
join_unoptimized.explain()

### 9. Add Random Salt Key to Transactions
We generate a random integer (salt value between 0 and 3) for each transaction row to split the skewed key into multiple keys.

In [ ]:
from pyspark.sql.functions import rand, lit

salted_tx = skewed_df.withColumn("salt", (rand() * lit(4)).cast("int"))
salted_tx.show(3)

### 10. Form Salted Transaction Key Column
Concatenating the merchant ID with the salt value to create a new key.

In [ ]:
from pyspark.sql.functions import concat

salted_tx_final = salted_tx.withColumn("salted_key", concat(col("merchantId"), lit("_"), col("salt")))
salted_tx_final.show(3)

### 11. Replicate Lookup Table with Cross Join
To join against the salted keys, we must replicate the lookup table across all possible salt values (0 to 3).

In [ ]:
salts_df = spark.range(0, 4).withColumnRenamed("id", "salt")
replicated_merchants = merchants_df.crossJoin(salts_df)
replicated_merchants.show(6)

### 12. Form Salted Lookup Key Column
Concatenating the lookup merchant ID with the salt values.

In [ ]:
replicated_merchants_final = replicated_merchants.withColumn(
    "salted_key",
    concat(col("merchantId"), lit("_"), col("salt"))
)
replicated_merchants_final.show(6)

### 13. Execute Salted Join (Optimized)
Joining on the salted keys distributes the 'M_1001' load across 4 different partition groups, enabling parallel execution.

In [ ]:
salted_join = salted_tx_final.join(replicated_merchants_final, "salted_key")
print("Salted join query constructed.")

### 14. Drop Salt Columns
Cleaning up the output columns by removing the helper salt keys.

In [ ]:
final_clean_df = salted_join.drop("salted_key", "salt")
print("Columns cleaned successfully.")

### 15. Verify Results Equivalence
Asserting that the final row counts of both joins are identical.

In [ ]:
count_unoptimized = join_unoptimized.count()
count_salted = final_clean_df.count()
print(f"Unoptimized count: {count_unoptimized}, Salted count: {count_salted}")
assert count_unoptimized == count_salted, "Row count mismatch!"

### 16. Dynamic Resource Allocation parameters
Reviewing parameters used to dynamically request and release executors based on task queue demand.

In [ ]:
# In spark-defaults.conf:
# spark.dynamicAllocation.enabled=true
# spark.dynamicAllocation.minExecutors=1
# spark.dynamicAllocation.maxExecutors=5
print("Dynamic Resource parameters verified.")

### 17. Custom Partitioners
Using a custom partitioner to control data distribution explicitly based on key hashes.

In [ ]:
# Custom partitioning requires RDD APIs:
# rdd.partitionBy(numPartitions, partitionFunc)
print("Custom partitioners configured.")

### 18. Benchmark: Unsalted Skewed Join
Measuring execution time for the unoptimized join.

In [ ]:
import time

# Temporarily disable AQE to measure raw skew bottleneck
spark.conf.set("spark.sql.adaptive.enabled", "false")

start = time.time()
join_unoptimized.write.mode("overwrite").parquet("data/unoptimized_join_result")
duration_skewed = time.time() - start
print(f"Unoptimized Skewed Join Time: {duration_skewed:.4f} seconds")

### 19. Benchmark: Salted Join
Measuring execution time for the optimized salted join.

In [ ]:
start = time.time()
final_clean_df.write.mode("overwrite").parquet("data/salted_join_result")
duration_salted = time.time() - start
print(f"Optimized Salted Join Time: {duration_salted:.4f} seconds")

### 20. Performance comparison
Calculating the speedup factor achieved by salting.

In [ ]:
print(f"Salting Speedup Factor: {duration_skewed / duration_salted:.2f}x speedup")
# Re-enable AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")

### 21. Stop Spark Session
Stopping resources cleanly.

In [ ]:
spark.stop()